# Audio Layering & Control — Mixing Sound in the Editor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/editor/feature/audio_asset_layering.ipynb)

---

Sound is half the story. A great video with poor audio loses impact immediately.

In this notebook, we'll explore how the VideoDB Editor handles audio—from basic volume control to sophisticated multi-track mixing. You'll learn how to mute video audio, layer background music, add voiceovers, and balance multiple audio sources for professional-sounding compositions.

**What you'll learn:**

- How AudioAsset works and its key parameters (`id`, `start`, `volume`)
- How to control video audio independently using VideoAsset `volume`
- How to layer multiple audio sources using horizontal stacking
- The "double start" concept: trimming source audio vs. timing in the timeline
- Best practices for volume balancing across dialogue, music, and voiceovers


---

## 📦 Step 1: Installing VideoDB SDK

In [1]:
!pip -q install videodb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 1.0 MB/s eta 0:00:00


---

## 📦 Step 2: Connect to VideoDB

We'll establish a secure connection to VideoDB using your API key.

In [2]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")
os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()
print("✅ Connected to VideoDB securely!")

Please enter your VideoDB API Key: ··········
✅ Connected to VideoDB securely!


---

## 📦 Step 3: Connect to a collection

Collections are where all your media assets (videos, audio, images) are stored. We'll get the default collection for this session.

In [3]:
coll = conn.get_collection()
print(f"✅ Using collection: {coll.id}")

✅ Using collection: c-5256e90f-4619-4451-9d23-071c1930bd34


---

## 📦 Step 4: Upload a video with audio

Let's upload a video that contains original audio. This will serve as our base content.

**Tip:** Replace the placeholder URL with any video URL (YouTube, direct MP4, etc.). The video should have dialogue or natural sound to demonstrate audio control.

In [4]:
# Upload video with original audio
video = coll.upload(url="https://www.youtube.com/watch?v=mLwlGsRhNIU")
print(f"✅ Video uploaded: {video.id}")

# If re-running this notebook, you can skip the upload and use an existing video:
# video = coll.get_video("your_video_id_here")

✅ Video uploaded: m-z-019edf17-3c4a-77e0-a88d-0c841fd2b0e6


---

## 📦 Step 5: Upload audio assets

Now we'll upload two separate audio files:

1. **Background music** — a music track to play underneath the video
2. **Voiceover** — a narration or spoken audio to overlay on the video

**Tip:** Replace these placeholder URLs with your own audio files (MP3, WAV, or any supported audio format). You can upload from direct URLs or use YouTube audio.

In [5]:
from videodb import MediaType
# Upload background music
music = coll.upload(url="https://www.youtube.com/watch?v=5kbpaZeRNQs", media_type=MediaType.audio)
print(f"✅ Music uploaded: {music.id}")

# If re-running:
# music = coll.get_audio("your_audio_id_here")

✅ Music uploaded: a-z-019edf18-2796-7773-9945-00dc7a826b6f


In [6]:
from videodb import MediaType
# Upload voiceover audio
voiceover = coll.upload(url="https://www.youtube.com/shorts/0UyOAtjKXpM", media_type=MediaType.audio)
print(f"✅ Voiceover uploaded: {voiceover.id}")

# If re-running:
# voiceover = coll.get_audio("your_audio_id_here")

✅ Voiceover uploaded: a-z-019edf18-bd08-7ca2-8978-8c05a596b880


---

## 📦 Step 6: Import Editor building blocks

We'll import the core Editor objects we need for this notebook:

- **Timeline** — the main canvas for our composition
- **Track** — the layer where we place clips
- **Clip** — the container that wraps assets with duration and effects
- **VideoAsset** — represents video files (with `volume` control for their audio)
- **AudioAsset** — represents pure audio files (music, voiceovers, SFX)
- **play_stream** — plays the generated video stream

In [7]:
from videodb import play_stream
from videodb.editor import Timeline, Track, Clip, VideoAsset, AudioAsset

---

## 📦 Step 7: Baseline — Video with original audio

Let's start with the simplest case: a video playing with its original audio at full volume.

**What's happening:**
- We create a VideoAsset with `volume=1.0` (original volume)
- The video plays for 10 seconds with its natural sound
- This establishes our baseline for comparison

**Key insight:** By default, video audio plays at full volume (1.0). We can adjust this using the `volume` parameter.

In [8]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=1.0
    ),
    duration=10
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"🎬 Baseline video with original audio: {stream_url}")
play_stream(stream_url)

🎬 Baseline video with original audio: https://play.videodb.io/v1/b2cc0240-3d4e-4130-aeb0-f15278d2efd5.m3u8


**What we see:** The video plays with its original audio at normal volume. This is our reference point.

---

## 📦 Step 8: Muting video audio

Sometimes you want the visual content but not the original sound. This is common when replacing dialogue, adding a different soundtrack, or creating silent background visuals.

**What's happening:**
- We set `volume=0` on the VideoAsset
- The video plays visually, but produces no sound
- This gives us a clean slate for adding other audio

**When to mute:** Use `volume=0` when you're replacing the original audio entirely with voiceover, music, or both.

In [19]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=0
    ),
    duration=10
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"🔇 Video with muted audio: {stream_url}")
play_stream(stream_url)

🔇 Video with muted audio: https://play.videodb.io/v1/3a834c16-7d11-4b5c-84b5-d4698e62398f.m3u8


**What we see:** The video plays normally, but you hear silence. The audio has been completely removed.

---

## 📦 Step 9: Adding background music

Now let's introduce AudioAsset—the object that represents pure audio files.

**What's happening:**
- We create an AudioAsset pointing to our uploaded music
- We set `volume=0.3` (30% of original volume) to keep it in the background
- We add both video and audio clips to the **same track at the same start time** (horizontal stacking)
- Both play simultaneously, and their audio **mixes together**

**Key insight:** Audio is additive. When multiple audio sources play at the same time, they combine. Volume balancing is critical to prevent one source from overpowering another.

In [10]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Video clip with original audio at full volume
video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=1.0
    ),
    duration=10
)

# Background music at 10% volume
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        volume=0.1
    ),
    duration=10
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, music_clip)  # Same start time = plays simultaneously
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"🎵 Video with background music: {stream_url}")
play_stream(stream_url)

🎵 Video with background music: https://play.videodb.io/v1/bdbddd4a-76d9-48cb-b366-8f5ffccec515.m3u8


**What we see:** The video plays with its original audio, and background music plays underneath at a lower volume. The two audio sources mix together naturally.

---

## 📦 Step 10: Muting video + adding background music

Let's combine what we've learned: mute the video's original audio and replace it with music.

**What's happening:**
- Video audio is set to `volume=0` (silent)
- Music audio is set to `volume=0.8` (80% volume, louder than before since there's no competing dialogue)
- This creates a clean audio replacement scenario

**Use case:** Music videos, montages, or any content where the original audio isn't needed.

In [11]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Muted video
video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=0
    ),
    duration=10
)

# Music at 80% volume
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        volume=0.8
    ),
    duration=10
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, music_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"🎵 Muted video with music replacement: {stream_url}")
play_stream(stream_url)

🎵 Muted video with music replacement: https://play.videodb.io/v1/3cbd2f31-f43b-49bf-b4f0-277422edbe44.m3u8


**What we see:** The video plays visually, but the only audio we hear is the background music. The original video sound is completely gone.

---

## 📦 Step 11: Adding voiceover over muted video

Voiceovers are a common use case: narration over visuals. Let's add a voiceover to our muted video.

**What's happening:**
- Video is muted (`volume=0`)
- Voiceover plays at full volume (`volume=1.0`) for clarity
- This creates a classic narration-over-video pattern

**When to use:** Explainer videos, tutorials, documentaries, or any content where spoken narration drives the story.

In [12]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Muted video
video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=0
    ),
    duration=10
)

# Voiceover at full volume
voiceover_clip = Clip(
    asset=AudioAsset(
        id=voiceover.id,
        volume=1.0
    ),
    duration=10
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, voiceover_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"🎙️ Muted video with voiceover: {stream_url}")
play_stream(stream_url)

🎙️ Muted video with voiceover: https://play.videodb.io/v1/9f4043ef-7e7e-4120-8502-dc6d20af3a98.m3u8


**What we see:** The video plays silently while the voiceover narration plays clearly. This is a clean voice-over-video setup.

---

## 📦 Step 12: Layering voiceover + background music

Let's create a professional audio mix by layering voiceover and background music together (both over muted video).

**What's happening:**
- Video is muted (`volume=0`)
- Voiceover plays at full volume (`volume=1.0`) for clarity
- Music plays quietly in the background (`volume=0.2`, just 20%)
- All three clips are added to the same track at the same time

**Key insight:** When layering voice with music, the music should be **much quieter** (typically 20-30% volume) so it doesn't compete with the narration.

In [13]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

# Muted video
video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=0
    ),
    duration=10
)

# Voiceover at full volume (primary audio)
voiceover_clip = Clip(
    asset=AudioAsset(
        id=voiceover.id,
        volume=1.0
    ),
    duration=10
)

# Background music at low volume (supports but doesn't compete)
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        volume=0.1
    ),
    duration=10
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, voiceover_clip)
track.add_clip(0, music_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"🎙️🎵 Professional mix: voiceover + background music: {stream_url}")
play_stream(stream_url)

🎙️🎵 Professional mix: voiceover + background music: https://play.videodb.io/v1/b6129e14-5aaf-4d30-88cd-14f4c45f01e5.m3u8


**What we see:** The video plays with clear voiceover narration, supported by quiet background music. This is the classic professional audio mix pattern.

---

## 📦 Step 13: The "Double Start" — Trimming vs. Timing

This is a crucial concept that applies to both VideoAsset and AudioAsset.

**There are two different "start" parameters:**

1. **AudioAsset(start=N)** — **TRIMMING**: Skip the first N seconds of the source audio file
2. **track.add_clip(start=N, clip)** — **TIMING**: Delay when the clip appears in the final timeline

Let's demonstrate both.

### Trimming the source audio

Sometimes you want to use a portion of an audio file, starting from the middle. The `start` parameter on AudioAsset trims the beginning.

**What's happening:**
- We set `AudioAsset(start=5)` on the music
- This skips the first 5 seconds of the music file
- The music starts playing from the 5-second mark of the source file
- It begins immediately at timeline position 0:00

In [14]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=0
    ),
    duration=10
)

# Music starts from 5-second mark of the source file
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        start=5,  # Skip first 5 seconds of the music file
        volume=0.5
    ),
    duration=10
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, music_clip)  # Starts at 0:00 in timeline
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"✂️ Music trimmed (starts from 5s of source file): {stream_url}")
play_stream(stream_url)

✂️ Music trimmed (starts from 5s of source file): https://play.videodb.io/v1/4b260e95-b7f3-49c6-9f5b-502e9e01725d.m3u8


**What we see:** The music you hear is from the middle of the music file (starting at 5 seconds in), but it begins playing immediately when the video starts.

### Delaying audio in the timeline

Now let's delay when the audio starts playing in the final video. This is controlled by `track.add_clip(start=N, clip)`.

**What's happening:**
- Video starts at timeline position 0:00
- Music is added at timeline position 3:00 (3 seconds in)
- The first 3 seconds have only video audio, then music fades in
- The music plays from its beginning (no trimming)

In [15]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=1.0
    ),
    duration=10
)

# Music plays from beginning of source file
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        volume=0.4
    ),
    duration=7
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(3, music_clip)  # Music starts at 3-second mark of timeline
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"⏱️ Music delayed (starts at 3s in timeline): {stream_url}")
play_stream(stream_url)

⏱️ Music delayed (starts at 3s in timeline): https://play.videodb.io/v1/a863df54-a700-497a-ab46-716b32352b59.m3u8


**What we see:** The video starts with only its original audio. At the 3-second mark, background music begins playing and continues for 7 seconds.

### Combining both: Trim AND delay

You can use both parameters together for precise control.

**Example scenario:** You want to skip the first 5 seconds of a song and have it begin 3 seconds into your final video.

**What's happening:**
- `AudioAsset(start=5)` — Skip the first 5 seconds of the music file
- `track.add_clip(3, music_clip)` — Start playing at the 3-second mark of the timeline
-  Result: The music starts at 3 seconds into your final video, playing from the 5-second mark of the source file

In [16]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=0.5
    ),
    duration=15
)

# Music: skip first 5s of source, start at 3s in timeline, play for 12s
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        start=5,  # TRIM: Skip first 5 seconds of music file
        volume=0.5
    ),
    duration=12
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(3, music_clip)  # TIMING: Start at 3s in timeline
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"✂️⏱️ Music trimmed AND delayed: {stream_url}")
play_stream(stream_url)

✂️⏱️ Music trimmed AND delayed: https://play.videodb.io/v1/3de018c3-3bf8-45c4-8c85-e8941b4961a2.m3u8


**What we see:** The video plays for 3 seconds with only its audio. At 3 seconds, music begins—but it's playing from the 5-second mark of the music file, continuing for 12 seconds.

---

## 📦 Step 14: Volume balancing best practices

Let's demonstrate the importance of proper volume balancing by comparing poor vs. good mixing.

**The golden rule:** Background music should **support** dialogue/voiceover, not compete with it.

### ❌ Poor balance: Music too loud

This is a common mistake—background music at the same volume as dialogue makes both hard to understand.

**What's happening:**
- Video audio: `volume=1.0` (full volume)
- Music: `volume=1.0` (also full volume)
- Voiceover: `volume=0.5` (half volume)
- Result: The music drowns out the voiceover, creating audio chaos

In [17]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=1.0
    ),
    duration=10
)

# Music too loud - competing with dialogue
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        volume=1.0  # ❌ Too loud!
    ),
    duration=10
)

voiceover_clip = Clip(
    asset=AudioAsset(
        id=voiceover.id,
        volume=0.5  # ❌ Too low!
    ),
    duration=10
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, music_clip)
track.add_clip(0, voiceover_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"❌ Poor mix (music too loud): {stream_url}")
play_stream(stream_url)

❌ Poor mix (music too loud): https://play.videodb.io/v1/970d17f1-21bc-4fdb-afb6-b757402e67e0.m3u8


**What we hear:** A muddy, chaotic mix where it's hard to distinguish dialogue from music. This sounds unprofessional.

### ✅ Good balance: Music as background

Now let's apply proper volume balancing principles.

**What's happening:**
- Voiceover audio: `volume=1.0` (dialogue is clear and prominent)
- Music: `volume=0.10` (10%, stays in the background)
- Result: Clean, professional audio where dialogue is clear and music provides atmosphere

**Volume guidelines:**
- **Dialogue/Voiceover:** 0.8 - 1.0 (80-100%)
- **Background music:** 0.1 - 0.3 (10-30%)
- **Sound effects:** 0.5 - 0.8 (50-80%, depending on importance)

In [18]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

video_clip = Clip(
    asset=VideoAsset(
        id=video.id,
        volume=0
    ),
    duration=10
)

# Music properly balanced - supports without competing
music_clip = Clip(
    asset=AudioAsset(
        id=music.id,
        volume=0.10  # ✅ Perfect background level
    ),
    duration=10
)

voiceover_clip = Clip(
    asset=AudioAsset(
        id=voiceover.id,
        volume=1.0
    ),
    duration=10
)

track = Track()
track.add_clip(0, video_clip)
track.add_clip(0, music_clip)
track.add_clip(0, voiceover_clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"✅ Good mix (music balanced): {stream_url}")
play_stream(stream_url)

✅ Good mix (music balanced): https://play.videodb.io/v1/da2e1cd5-f52a-490f-8014-646a041118f2.m3u8


**What we hear:** Clear voiceover with pleasant background music that enhances the video without distracting. This is professional audio mixing.

---

## ✅ Wrap-up

🎉 **You've mastered audio layering in the VideoDB Editor!**

### What We Covered

1. **AudioAsset basics** — The three key parameters: `id`, `start`, and `volume`
2. **Video audio control** — Using `volume` on VideoAsset to adjust or mute original audio
3. **Audio layering** — Adding multiple audio sources to the same track at the same time (horizontal stacking)
4. **The "double start" concept** — Distinguishing between trimming source audio (`AudioAsset.start`) and timing in the timeline (`track.add_clip(start, clip)`)
5. **Volume balancing** — Professional mixing principles to keep dialogue clear and music supportive

### Key Insights

✅ **Audio is additive** — Unlike video (where tracks obscure each other), all audio playing at the same time mixes together

✅ **Volume control is critical** — Use the range 0.0 (mute) to 2.0 (double volume), with 1.0 as the baseline. Background music should typically be 20-30% of dialogue volume.

✅ **AudioAsset has no visual component** — It only contributes sound. Duration is controlled by the Clip wrapper, not the asset itself.

✅ **Horizontal stacking creates simultaneous playback** — Add clips to the same track at the same start time to layer audio sources.

### Practical Volume Guidelines

- **Dialogue/Voiceover:** 0.8 - 1.0 (primary audio, must be clear)
- **Background music:** 0.2 - 0.4 (supports without competing)
- **Sound effects:** 0.5 - 0.8 (noticeable but not overwhelming)

### Actionable Next Steps

🔹 **Experiment with different volume ratios** to find the perfect balance for your content style

🔹 **Try layering multiple music tracks** at different volumes to create rich soundscapes

🔹 **Explore audio trimming** to use specific sections of longer audio files (intros, choruses, bridges)

🔹 **Add sound effects** at precise moments using timing control to punctuate visual actions

🔹 **Combine audio with other Editor features** like Transitions, Filters, and CaptionAssets for complete compositions

---

Audio is half the story—sometimes more than half. With these tools, you can create professional, balanced audio mixes that elevate your video content. Happy editing! 🎬🎵